In [1]:
import pandas as pd
import numpy as np
from datetime import timedelta
import glob
import re
import os

In [2]:
%load_ext autoreload

In [19]:
#Run this to reload the python file
%autoreload 2
from utils import *

### Importing Data

In [4]:
# path file
paths = glob.glob('./Data/IMN_raw/*IMN*csv')

In [5]:
# example
df = pd.read_csv(paths[0])

### Checking missing dates

In [6]:
start_date = '2001-01-01 00:00:00'
end_date = '2022-12-31 23:00:00'

In [7]:
df = df[(df['date'] >= start_date) & (df['date'] <= end_date)]

In [9]:
m_dates = missing_dates(df, 'date', start_date, end_date, 'H')

In [13]:
# create a DataFrame with missing dates and 'NA' in the 'pcp' column
m_dates = pd.DataFrame({'station_number': paths[0][15:-8], 'date': m_dates, 'pcp': np.nan})

In [14]:
df = pd.concat([df, m_dates], ignore_index=True)

In [15]:
df['date'] = pd.to_datetime(df['date'])  

In [16]:
df = df.sort_values(by='date')

### Replace missing values

In [17]:
df = missing_values(df, 'pcp', -9)

### Calculate percentage of missing data

In [20]:
perc = nan_percentage(df, 'date', 'pcp', start_date, end_date, 'H')

## For many files

In [ ]:
start_date = '2001-01-01 00:00:00'
end_date = '2022-12-31 23:00:00'

In [23]:
percentage = []

for path in paths:
    
    df = pd.read_csv(path)
    
    df = df[(df['date'] >= start_date) & (df['date'] <= end_date)]
    
    # checking missing dates
    m_dates = missing_dates(df, 'date', start_date, end_date, 'H')
    
    # create a DataFrame with missing dates and 'NA' in the 'pcp' column
    m_dates = pd.DataFrame({'station_number': path[15:-8], 'date': m_dates, 'pcp': np.nan})
    
    df = pd.concat([df, m_dates], ignore_index=True)
    
    df['date'] = pd.to_datetime(df['date'])  
    
    df = df.sort_values(by='date')
    
    # replace missing values to nan
    df = missing_values(df, 'pcp', -9)
    
    #calculate percentage of missing data in a specific time range
    perc = nan_percentage(df, 'date', 'pcp', start_date, end_date, 'H')
    percentage.append(perc)
    
    if perc < 10:
        df.to_csv(f"./Data/harmonized/{path[15:-8]}_imn.csv")

In [24]:
numbers = []
for path in paths:
    number = path[15:-8]
    numbers.append(number)

In [27]:
tmp = pd.DataFrame()
tmp['station_number'] = numbers
tmp['percentage'] = percentage

In [28]:
# adding the percentage values into the metadata file
meta = pd.read_csv('./Data/metadata/IMN_stations.csv')

In [30]:
meta['Número'] = meta['Número'].astype(str)
merged_df = meta.merge(tmp, left_on='Número', right_on='station_number')

In [31]:
merged_df.to_csv('./Data/metadata/IMN_stations_rew.csv')

### Join all files into one

In [33]:
# path file
paths = glob.glob('./Data/harmonized/*imn*csv')

In [34]:
df_comb = pd.DataFrame()

In [35]:
df_comb['date'] = pd.date_range(start=start_date, end=end_date, freq='H')

In [36]:
for path in paths:
    df = pd.read_csv(path)
    
    df_comb[path[18:-8]] = df['pcp']

In [37]:
df_comb.to_csv('./Data/harmonized/unif_IMN.csv', index=False)